In [0]:
%pip install python-dotenv --quiet
dbutils.library.restartPython()

In [0]:
# Conectando via abfss a Tabela food_avaliacoes_produto.csv
import os
from dotenv import load_dotenv

load_dotenv("/Workspace/Users/ik.kukoo@gmail.com/.env", override=True)

STORAGE_ACCOUNT = "internshipdatalake"

adls_options = {
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        os.getenv("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        os.getenv("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}

PATH_AVALIACOES = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/batch-data/food_avaliacoes_produto.csv"
print("✅ Config OK →", PATH_AVALIACOES)

In [0]:
# Leitura da Tabela food_avaliacoes_produto.csv
df = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(PATH_AVALIACOES)
)

print(f"Linhas: {df.count()} | Colunas: {len(df.columns)}")
df.printSchema()

In [0]:
# Amostra da Tabela
df.show(10, truncate=False)

In [0]:
# Estatísticas Descritivas da Tabela
df.describe().show(truncate=False)

In [0]:
# Análise de Nulos da Tabela
from pyspark.sql.functions import col, sum as spark_sum

nulls = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
print("🔍 Nulos por coluna:")
nulls.show(truncate=False)

In [0]:
# Distribuições das Avaliações
from pyspark.sql.functions import count, round as spark_round

total = df.count()

df.groupBy("nota").agg(
    count("*").alias("quantidade")
).withColumn(
    "percentual",
    spark_round((col("quantidade") / total) * 100, 2)
).orderBy("nota").show()
from pyspark.sql.functions import count, round as spark_round

total = df.count()

df.groupBy("nota").agg(
    count("*").alias("quantidade")
).withColumn(
    "percentual",
    spark_round((col("quantidade") / total) * 100, 2)
).orderBy("nota").show()

In [0]:
# Média de Nota por SKU
from pyspark.sql.functions import avg, count

nota_por_sku = df.groupBy("sku").agg(
    avg("nota").alias("media_nota"),
    count("*").alias("total_avaliacoes")
).filter(col("total_avaliacoes") >= 10)  # mínimo 10 avaliações para ser relevante

print("🏆 Top 10 SKUs melhor avaliados:")
nota_por_sku.orderBy(col("media_nota").desc()).show(10, truncate=False)

print("💔 Top 10 SKUs pior avaliados:")
nota_por_sku.orderBy(col("media_nota").asc()).show(10, truncate=False)

In [0]:
# Avaliações verificadas vs não verificadas
df.groupBy("verificada").count().show()

In [0]:
# Evolução temporal das avaliações
from pyspark.sql.functions import to_date, date_format

df.withColumn("mes", date_format("dt_avaliacao", "yyyy-MM")) \
  .groupBy("mes").count() \
  .orderBy("mes") \
  .show(30, truncate=False)